# Slide-level classification · tile encoder + MIL

Predict a slide-level label from a whole-slide image, the standard MIL way:

```
Dataset -> FeatureExtractor (tile & encode) -> train (aggregator + head) -> evaluate
```

A frozen **tile encoder** embeds each tile into a *bag* of vectors, and a
trainable **MIL aggregator** pools the bag into one slide vector before the
head. The other route — a slide-native encoder that returns the slide vector
directly, with no aggregator — is the
[slide encoder walkthrough](walkthrough-slide-encoder.ipynb). For per-tile
labels with no bag at all, see the
[tile-level walkthrough](walkthrough-tile-level.ipynb).

> Tiny synthetic data, CPU-only, ungated encoder — the numbers are
> meaningless; the point is the API.

## ⚠️ Scaffolding (not soma API)

The cell below fabricates toy slides and the two CSVs soma expects. **Replace
this with your own slides and labels** — only the on-disk contract matters:

* `dataset.csv` — one row per slide: `sample_id`, `image_path`, `label`.
* `splits.csv` — `sample_id`, `split` (`train` / `tune` / `test*`), optional
  `fold`.

In [ ]:
import logging, warnings
warnings.filterwarnings('ignore')
logging.getLogger().setLevel(logging.ERROR)

import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import tifffile

WORK = Path(tempfile.mkdtemp(prefix='soma-slide-mil-'))
SLIDES = WORK / 'slides'; SLIDES.mkdir()
rng = np.random.default_rng(0)

def make_toy_slide(path, size=640):
    """A white background with a central H&E-ish blob, saved as a tiled TIFF
    whose resolution tags make OpenSlide report 0.5 microns/pixel."""
    img = np.full((size, size, 3), 240, np.uint8)
    yy, xx = np.mgrid[0:size, 0:size]
    blob = ((xx - size // 2) ** 2 + (yy - size // 2) ** 2) < (size * 0.35) ** 2
    tissue = np.stack([np.full((size, size), 150),
                       np.full((size, size), 70),
                       np.full((size, size), 160)], -1).astype(np.int16)
    tissue += rng.integers(-30, 30, (size, size, 3))
    img[blob] = np.clip(tissue, 0, 255).astype(np.uint8)[blob]
    tifffile.imwrite(path, img, photometric='rgb', tile=(256, 256),
                     resolution=(20000, 20000), resolutionunit='CENTIMETER')

N = 8
sample_ids = [f's{i:02d}' for i in range(N)]
for sid in sample_ids:
    make_toy_slide(SLIDES / f'{sid}.tif')

# train/tune/test assignment (single fold) with both classes in every split
split = (['train'] * 4) + (['tune'] * 2) + (['test'] * 2)
binary = [0, 1, 0, 1,  0, 1,  0, 1]

dataset_csv = WORK / 'dataset.csv'
splits_csv = WORK / 'splits.csv'
pd.DataFrame({'sample_id': sample_ids,
              'image_path': [str(SLIDES / f'{s}.tif') for s in sample_ids],
              'label': binary}).to_csv(dataset_csv, index=False)
pd.DataFrame({'sample_id': sample_ids, 'split': split}).to_csv(splits_csv, index=False)

print(pd.read_csv(dataset_csv).head().to_string(index=False))

## 1. Load the dataset and splits

`Dataset` reads `dataset.csv` and infers the label space; `Splits` pairs
`splits.csv` with it. soma never repartitions your data — the splits you
provide are the splits it uses, which keeps evaluation reproducible and
leakage-free.

In [ ]:
from soma import Dataset, Splits

dataset = Dataset(dataset_csv)
splits = Splits(splits_csv, dataset)
print('slides:', len(dataset.sample_ids), '| folds:', splits.num_folds)

## 2. Extract a bag of tile features per slide

`FeatureExtractor` runs preprocessing (tissue masking + tiling) and a frozen
tile encoder over each tile, writing one **bag of vectors per slide**. We use
[phikon](https://huggingface.co/owkin/phikon) — ungated and small enough for
CPU. `CacheConfig(enabled=True)` caches the bags so repeat experiments reuse
this one extraction.

In [ ]:
from soma import FeatureExtractor, EncoderConfig, PreprocessingConfig, CacheConfig

extractor = FeatureExtractor(
    dataset,
    EncoderConfig(name='phikon'),
    preprocessing=PreprocessingConfig(
        backend='openslide',
        requested_tile_size_px=224,
        requested_spacing_um=0.5,
        tissue_method='otsu',
        # Toy slides are small; a fine segmentation downsample + low area
        # threshold keep the tissue mask usable (defaults assume real WSIs).
        seg_downsample=16,
        a_t=1,
    ),
    cache=CacheConfig(enabled=True, root_dir=str(WORK / 'cache')),
    output_root=str(WORK / 'output'),
)
store = extractor.extract(feature_dir='features')
print('feature bags for', len(store.available_samples), 'slides')

## 3. Train the aggregator + head

`train()` consumes the bag store and trains a MIL **aggregator** (here
attention-MIL, `abmil`) plus a **task head**. The aggregator pools each
slide's bag into one vector; the head maps it to a prediction.
`TaskConfig('binary_classification')` picks the head, loss, and metrics.

Because features don't depend on the labels, **multiclass, regression, and
survival are the same call with a different `TaskConfig` on this same store** —
see the [task heads guide](../tasks.rst).

In [ ]:
from soma import AggregatorConfig, TaskConfig, TrainingConfig, EvalConfig, train

result = train(
    feature_store=store,
    dataset=dataset,
    splits=splits,
    aggregator=AggregatorConfig(name='abmil'),
    task=TaskConfig(name='binary_classification'),
    training=TrainingConfig(epochs=3, learning_rate=1e-3, batch_size=4, seed=0),
    evaluation=EvalConfig(metrics=['balanced_accuracy', 'auroc']),
    run_dir=str(WORK / 'runs' / 'binary'),
)
print('run dir:', result.run_dir)

## 4. The one-shot `Pipeline` equivalent

Everything above collapses into a single config-driven `Pipeline` call. The
building blocks are for reusing features across experiments; the `Pipeline`
is for when you just want the result.

*(Shown for reference, not executed.)*

```python
from soma import (
    Pipeline, PipelineConfig, PreprocessingConfig, EncoderConfig,
    AggregatorConfig, TaskConfig, TrainingConfig, EvalConfig, CacheConfig,
)

config = PipelineConfig(
    dataset_csv=str(dataset_csv),
    splits_csv=str(splits_csv),
    output_root='output/binary',
    dataset_type='slide',
    preprocessing=PreprocessingConfig(
        backend='openslide', requested_tile_size_px=224, requested_spacing_um=0.5,
        tissue_method='otsu',
    ),
    encoder=EncoderConfig(name='phikon'),
    aggregator=AggregatorConfig(name='abmil'),
    task=TaskConfig(name='binary_classification'),
    training=TrainingConfig(epochs=3, learning_rate=1e-3, batch_size=4),
    evaluation=EvalConfig(metrics=['balanced_accuracy', 'auroc']),
    cache=CacheConfig(enabled=True),
)
results = Pipeline(config).run()
```